In [0]:
%run ../config_setup

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr

spark = SparkSession.builder.getOrCreate()

print("🚀 Building gold business dimensions...")

# ============================================================
# 1. DIM LOCATION
# ============================================================
site_std = spark.read.table(f"{SILVER}.site_standardized")
site_std.select(
    expr("xxhash64(std_city, std_country)").alias("location_sk"), 
    col("std_city"), 
    col("std_state"), 
    col("std_country"), 
    col("is_standardized")
).dropDuplicates(["location_sk"]) \
 .write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.dim_location")

print("  ✓ dim_location done")

# ============================================================
# 2. DIM SITE
# ============================================================
silver_site = spark.read.table(f"{SILVER}.site").select("site_id", "geo_lat", "geo_lon")
dim_site = site_std.join(silver_site, "site_id", "left").select(
    expr("xxhash64(site_id)").alias("site_sk"), 
    col("site_id"), 
    col("site_name"), 
    col("geo_lat"), 
    col("geo_lon"), 
    expr("xxhash64(std_city, std_country)").alias("location_sk")
)
dim_site.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.dim_site")

print("  ✓ dim_site done")

# ============================================================
# 3. DIM OWNER
# ============================================================
owner_df = spark.read.table(f"{SILVER}.owner")

owner_df.select(
    expr("xxhash64(owner_id)").alias("owner_sk"), 
    col("owner_id"), 
    col("owner_name")
).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.dim_owner")

print("  ✓ dim_owner done")

# ============================================================
# 4. DIM OPERATOR
# ============================================================
spark.read.table(f"{SILVER}.operator").select(
    expr("xxhash64(operator_id)").alias("operator_sk"), 
    col("operator_id"), 
    col("operator_name")
).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.dim_operator")

print("  ✓ dim_operator done")

print("✅ gold_03_dim_business built successfully!")

In [0]:
rows = spark.sql(f"""
SELECT table_name, table_type
FROM {CATALOG}.information_schema.tables
WHERE table_schema = '{GOLD_SCHEMA}'
ORDER BY table_name
""").collect()
print(f"TABLES IN {GOLD}:")
for r in rows:
    print(f"  {r.table_name} ({r.table_type})")
print("TOTAL:", len(rows))